## Imports

In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.ticker import PercentFormatter

## Extract dataframe from database

In [ ]:
DB_PATH = Path("database.sqlite")
OUTPUT_DIR = Path("cve_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)

if not DB_PATH.is_file():
    raise FileNotFoundError("Place database.sqlite in same directory as this notebook before running it.")

# Open database
conn = sqlite3.connect(DB_PATH)

tables=[]
for year in range(2022,2026):
    table = pd.read_sql_query(f"SELECT * FROM YEAR_{year}", conn)
    table["cve_year"] = year
    tables.append(table)

df = pd.concat(tables, ignore_index=True)

conn.close()

original_columns = df.columns.drop("cve_year")


## Dataset overview

In [ ]:
print(f"Records: {len(df):,}")
print(f"Original columns: {len(original_columns)}")
print(f"Duplicate CVE Ids: {df['cveId'].duplicated().sum():,}")
display(df.head())
display(df.groupby("cve_year").size().to_frame("Records"))

quality = pd.DataFrame({
    "Missing": df[original_columns].isna().sum().sum(),
    "Missing (%)": df[original_columns].isna().mean() * 100,
    "Distinct values": df[original_columns].nunique()
})

display(quality.round(2))


## Publication dates and placeholder labels

In [ ]:
publication_dates=pd.to_datetime(df["datePublished"], format="mixed", utc=True)
print(f"Publication dates: {publication_dates.min().date()} to {publication_dates.max().date()}")
display(pd.crosstab(df["cve_year"], publication_dates.dt.year))

placeholder_counts={}
for col in ["product", "vendor"]:
    values = df[col].str.strip().str.lower()
    placeholder_counts[col] = values.value_counts().reindex(["unknown", "n/a", "na"], fill_value=0)

display(pd.DataFrame(placeholder_counts).T)

## Conclusion 1: Medium and High severity are more frequent in this dataset

# Method: descriptive statistics and frequency analysis

We summarize class frequencies, mean and median scores, and the interquartile range. The severity bands are None $=0$, Low $=0.1-3.9$, Medium $=4.0-6.9$, High $=7.0-8.9$, and Critical $=9.0-10.0$.

These are descriptions, not a hypothesis test.

In [ ]:
display(df["baseScore"].describe().round(3))
q1, q3=df["baseScore"].quantile([0.25,0.75])
iqr=q3-q1

lower = round(q1 - 1.5 * iqr, 1)
upper = round(q3 + 1.5 * iqr, 1)
outliers = df[(df["baseScore"]<lower) | (df["baseScore"]>upper)]
print(f"Number of outliers: {len(outliers)}")

In [ ]:
# Derive severity labels
severity_order = ["None", "Low", "Medium", "High", "Critical"]
score = df["baseScore"]
df["severity_from_score"] = np.select(
    [score==0, score<4, score<7, score<9],
    ["None", "Low", "Medium", "High"],
    default="Critical"
    )
severity_counts= df["severity_from_score"].value_counts().reindex(severity_order)
severity_pct = 100* severity_counts / len(df)
display(pd.DataFrame({"Records": severity_counts, "Percent": severity_pct}).round(2))

label_mismatches = df.loc[
    df["baseSeverity"] != df["severity_from_score"],
    ["cveId", "baseScore", "baseSeverity", "severity_from_score"]
]

print("Records whose label is different from the score derived one:")
display(label_mismatches)

In [ ]:
fig, ax = plt.subplots(figsize=(11,6), layout="constrained")

ax.barh(severity_order, severity_pct, height=0.6)
ax.invert_yaxis()
ax.set_xlim(0,67)
ax.set_xlabel("Share of CVE records")
ax.xaxis.set_major_formatter(PercentFormatter(100))

ax.grid(axis="x")
ax.set_axisbelow(True)
ax.spines[["top","right","left"]].set_visible(False)

for i,(count,pct) in enumerate(zip(severity_counts, severity_pct)):
    ax.text(pct+1, i, f"{pct:.2f}% | {count:,}", va="center")

medium_high=severity_pct["Medium"]+severity_pct["High"]

fig.suptitle(
    f"Medium and High account for {medium_high:.1f}% of records",
    fontsize=19, weight="bold"
)
ax.set_title(
    f"Descriptive statistics | n = {len(df):,} | Median = {score.median():.1f}| Middle 50%: {q1:.1f}-{q3:.1f}"
)

plt.savefig(OUTPUT_DIR / "01_severity_distribution.png")
plt.show()


# Interpretation:
Medium and High together account for $85.86$% of records, with a median score of $6.7$. The severity distribution is imbalanced, with Medium alone representing $49.95$%
# Data quality:
`CVE-2022-3806` is incorrectly labeled Low despite a score of $9.8$. Additionally, $223$ low-score outliers were identified using the IQR method.
# Conclusion:
CVE severity is unevenly distributed, with Medium and High accounting for 85.86% of records. This imbalance should be considered when analyzing or comparing severity categories.




## Conclusion 2: Attack vectors differ across CVE-year groups
# Method: Chi-square test of independence

$H_0$: Attack vector and CVE year are independent
$H_1$: Attack vector and CVE year are associated

We compare the four attack vector categories across the four CVE identifier years. 

In [ ]:
vectors = ["Network", "Local", "Adjacent", "Physical"]
counts = pd.crosstab(df["cve_year"], df["attackVector"])[vectors]
percentages = counts.div(counts.sum(axis=1), axis=0) * 100

chi2, p, dof, expected = stats.chi2_contingency(counts)

print(f"p-value: {p:.3g} (very small might display as 0)")
display(percentages.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(11,6), layout="constrained")

for vector in vectors:
    ax.plot(percentages.index, percentages[vector], label=vector, marker="o")

for year in percentages.index:
    ax.annotate(f"{percentages.loc[year, 'Network']:.1f}%",
                (year, percentages.loc[year, "Network"]),
                xytext=(0,10),textcoords="offset points", ha="center")

ax.set_xticks(percentages.index)
ax.set_ylim(0,100)
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.set_xlabel("CVE identifier year")
ax.set_ylabel("Share of cohort")
ax.grid(axis="y")
ax.spines[["top","right","left"]].set_visible(False)
ax.legend(ncol=4,frameon=False)
fig.suptitle("Later CVE cohorts have a larger Network share", fontsize=18, weight="bold")

plt.savefig(OUTPUT_DIR / "02_attack_vector_by_cohort.png")
plt.show()


# Interpretation:
Network attack vectors increase from $64.48$% in the $2022$ cohort to $78.73$% in 2025. The p-value is less than $0.05$ so we reject $H_0$. Note that this pattern is in recorded vulnerabilities, not that network attacks became more frequent.